In [ ]:
from math import sqrt

def confidence(ups, downs):
    n = ups + downs

    if n == 0:
        return 0

    z = 1.96 #1.44 = 85%, 1.96 = 95%
    phat = float(ups) / n
    return ((phat + z*z/(2*n) - z * sqrt((phat*(1-phat)+z*z/(4*n))/n))/(1+z*z/n))

In [ ]:
from math import sqrt
from statistics import NormalDist

def wilson_cc(successes, trials, confidence=0.90):
    """Wilson score interval with continuity correction (Newcombe 1998).

    Returns (lower, upper). Wider and more conservative than the plain
    Wilson interval; coverage is at or above nominal for small n.
    """
    if trials <= 0:
        raise ValueError("trials must be positive")
    if not 0 <= successes <= trials:
        raise ValueError("successes must be between 0 and trials")
    if not 0 < confidence < 1:
        raise ValueError("confidence must be between 0 and 1")

    n = trials
    p_hat = successes / n
    q_hat = 1.0 - p_hat
    z = NormalDist().inv_cdf(1.0 - (1.0 - confidence) / 2.0)
    z2 = z * z
    denom = 2.0 * (n + z2)

    # Roots of (|p_hat - p| - 1/2n)^2 = z^2 * p(1-p)/n, with the degenerate
    # tails pinned to 0 / 1 (the radicand is not valid there).
    if successes == 0:
        lower = 0.0
    else:
        rad = z2 - 2.0 - 1.0 / n + 4.0 * p_hat * (n * q_hat + 1.0)
        lower = (2.0 * n * p_hat + z2 - 1.0 - z * sqrt(max(rad, 0.0))) / denom

    if successes == n:
        upper = 1.0
    else:
        rad = z2 + 2.0 - 1.0 / n + 4.0 * p_hat * (n * q_hat - 1.0)
        upper = (2.0 * n * p_hat + z2 + 1.0 + z * sqrt(max(rad, 0.0))) / denom

    return max(0.0, lower), min(1.0, upper)


def wilson_cc_lower_bound(successes, trials, confidence=0.95):
    """Conservative lower bound for ranking. Note this is the lower end of a
    two-sided interval, so at confidence=0.95 it is a one-sided 97.5% bound."""
    return wilson_cc(successes, trials, confidence)[0]


In [ ]:
import pickle


FILEPATH  = "./results/satml/level_XX_attacker_gpt_all.pickle"
OUTNAME = "./results/satml/SRB_confidence_intervals_level_XX.pickle"
# sorted_methods = ['presidio', 'azure', 'scrubadub', 'uniner', 'gliner', 'textwash', 'gpt_basic', 'gpt_clio', 'gpt_rescriber', 'llama_basic', 'llama_clio', 'llama_rescriber', 'gemini_basic', 'tem_eps11.0', 'pre_anon', 'anthropic_rescriber', 'privacy_filter', 'anthropic_basic', 'anthropic_full', 'anthropic_clio', 'eu_guardrail']
sorted_methods = ['presidio_srb', 'azure', 'scrubadub', 'uniner', 'gliner', 'textwash', 'gpt_basic', 'pre_anon',  'privacy_filter', 'anthropic_basic', 'eu_guardrail']
# N = 200
levels = ["1", "2", "3"]
data = dict()

for level in levels:
    for method in sorted_methods:
        trials = 0
        successes = 0
        threshold = 0.2
        
        filepath = FILEPATH.replace("XX", level)
        with open(filepath, "rb") as f:
            reid_data = pickle.load(f)
        if method == "tem_eps10.0" and "tem_eps10.0" not in reid_data['correctness']:
            method = "tem_eps11.0"
        for i, entry in enumerate(reid_data["correctness"][method]):
            if method == "tem_eps10.0" and "tem_eps10.0" not in reid_data['correct_direct_ids']:
                if len(reid_data['correct_direct_ids']['tem_eps11.0'][i] >= 1 or entry >= threshold):
                    successes+=1
            elif len(reid_data['correct_direct_ids'][method][i]) >= 1 or entry >= threshold:
                successes += 1
            trials += 1
        # if method == "tem_eps11.0":
        #     method = "tem_eps10.0"
        data[method] = data.get(method,dict())
        data[method]["successes"] = successes + data[method].get("successes", 0)
        data[method]["trials"] = trials + data[method].get("trials", 0)


    outdict = dict()
    for method in sorted_methods:
        successes = data[method]["successes"]
        trials = data[method]["trials"]
        lower,upper = wilson_cc(successes, trials)
        success_rate = successes/trials
        confidence_interval = ((upper - success_rate) + (success_rate-lower))/2
        print (method,  success_rate, confidence_interval)
        outdict[method] = dict()
        outdict[method]["reid_success_rate"] = success_rate
        outdict[method]["confidence_interval"] = confidence_interval
    outname = OUTNAME.replace("XX", level)
    with open(outname, 'wb') as handle:
        pickle.dump(outdict, handle, protocol=pickle.HIGHEST_PROTOCOL)



